In [ ]:
!pip install -q transformers torch pillow matplotlib opencv-python

import cv2
import urllib.request
import matplotlib.pyplot as plt
from PIL import Image
from transformers import pipeline

In [ ]:
from google.colab import files
import io
from PIL import Image

# Upload image from your computer
uploaded = files.upload()

# Load uploaded image
filename = list(uploaded.keys())[0]
img = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
from PIL import Image
import io

def take_photo():
    js = Javascript("""
    async function takePhoto() {
      try {
        const stream = await navigator.mediaDevices.getUserMedia({ video: true });

        const div = document.createElement('div');
        const video = document.createElement('video');
        const button = document.createElement('button');
        const status = document.createElement('div');

        button.textContent = 'Capture';
        status.textContent = 'Camera ready';
        status.style.margin = '10px 0';

        video.style.display = 'block';
        video.style.marginBottom = '10px';

        div.appendChild(status);
        div.appendChild(video);
        div.appendChild(button);
        document.body.appendChild(div);

        video.srcObject = stream;
        await video.play();

        await new Promise((resolve) => button.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(track => track.stop());
        div.remove();

        return canvas.toDataURL('image/jpeg', 0.95);
      } catch (err) {
        return "ERROR: " + err.name + ": " + err.message;
      }
    }
    """)
    display(js)
    data = eval_js("takePhoto()")

    if isinstance(data, str) and data.startswith("ERROR:"):
        raise RuntimeError(data)

    binary = b64decode(data.split(",")[1])
    return Image.open(io.BytesIO(binary)).convert("RGB")

img = take_photo()
img

In [ ]:


# img1 = "https://t3.ftcdn.net/jpg/02/43/12/34/360_F_243123463_zTooub557xEWABDLk0jJklDyLSGl2jrr.jpg"
# img2 = 'https://thumbs.dreamstime.com/z/woman-sad-face-crying-sad-expression-sad-emotion-despair-sadness-woman-emotional-stress-pain-woman-sitting-alone-th-61663852.jpg'
# img3 = "https://thumbs.dreamstime.com/b/sad-lonely-pensive-old-senior-woman-12781694.jpg"
# img4= "https://ichef.bbci.co.uk/images/ic/1024xn/p066st0k.jpg"

# # Download sample image
# urllib.request.urlretrieve(
#     img1,
#     "sample_face.png"
# )

# # Load image
# img = Image.open("sample_face.png").convert("RGB")

# Hugging Face image-classification pipeline
clf = pipeline(
    "image-classification",
    model="trpakov/vit-face-expression"
)

results = clf(img)

# Sort high to low
results = sorted(results, key=lambda x: x["score"], reverse=True)

labels = [r["label"] for r in results]
scores = [r["score"] for r in results]

# Plot
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.bar(labels, scores)
plt.title(f"Predicted Expression: {labels[0]}")
plt.xticks(rotation=45)
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

print("\nExpression Scores:")
for r in results:
    print(f"{r['label']}: {r['score']:.2%}")